# Riesgo de retraso (402): curva ROC e importancia de variables

Gráficos para el documento del proyecto. No entrena nada: usa el modelo publicado (`modelos/riesgo_retraso/actual.txt`) y el mismo 20 % de prueba que `entrenar.py` (misma semilla y división por obra).

Antes de abrirlo:

```sh
uv sync --group notebook
uv run python -m ml.riesgo.generar --obras 4000 --semilla 2026
```

> Estas métricas muestran que el modelo aprende las relaciones simuladas; no prueban que acierte con obras reales.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupShuffleSplit

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
MODELOS = RAIZ / "modelos" / "riesgo_retraso"
VERSION = (MODELOS / "actual.txt").read_text(encoding="utf-8").strip()
metricas = json.loads((MODELOS / VERSION / "metricas.json").read_text(encoding="utf-8"))
variables = [
    v["nombre"] for v in json.loads((MODELOS / VERSION / "variables.json").read_text(encoding="utf-8"))["variables"]
]
booster = xgb.Booster()
booster.load_model(str(MODELOS / VERSION / "modelo.ubj"))

datos = pd.read_csv(RAIZ / "ml" / "riesgo" / "datos" / "riesgo_sintetico.csv")
_, prueba = next(
    GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=metricas["semilla"]).split(datos, groups=datos["obra_id"])
)
prueba = datos.iloc[prueba]
y = prueba["se_retrasa"]
probabilidad = booster.predict(xgb.DMatrix(prueba[variables], feature_names=variables))
print(VERSION, metricas["datos"], len(prueba), "filas de prueba")
print(json.dumps(metricas["metricas"], indent=2))

## Curva ROC: modelo frente a la regla actual (`plazo − avance`)

In [ ]:
fpr_m, tpr_m, _ = roc_curve(y, probabilidad)
fpr_r, tpr_r, _ = roc_curve(y, prueba["brecha"])
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr_m, tpr_m, label=f"Modelo XGBoost (AUC {roc_auc_score(y, probabilidad):.3f})")
ax.plot(fpr_r, tpr_r, label=f"Regla plazo − avance (AUC {roc_auc_score(y, prueba['brecha']):.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Sensibilidad")
ax.set_title(f"Riesgo de retraso {VERSION}: curva ROC (20 % de prueba)")
ax.legend(loc="lower right")
plt.show()

## Importancia de las variables

In [ ]:
ganancia = pd.Series(booster.get_score(importance_type="gain")).reindex(variables).fillna(0)
contribuciones = booster.predict(xgb.DMatrix(prueba[variables], feature_names=variables), pred_contribs=True)[:, :-1]
media_abs = pd.Series(abs(contribuciones).mean(axis=0), index=variables)

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 5))
ganancia.sort_values().plot.barh(ax=izq, title="Ganancia promedio por división (XGBoost)")
media_abs.sort_values().plot.barh(ax=der, title="Contribución media |pred_contribs| (factores)")
plt.tight_layout()
plt.show()